In [4]:
import os
import sys

project_root = "/root/work/tvm-ansor"
os.environ["TVM_HOME"] = f"{project_root}"
os.environ["TVM_LIBRARY_PATH"] = f"{project_root}/build-release"
if f"{project_root}/python" not in sys.path:
    sys.path.insert(0, f"{project_root}/python")

sys.path = [p for p in sys.path if not p.startswith(f"{project_root}/build")]
sys.path.append(f"{project_root}/build-release")
os.environ["LD_LIBRARY_PATH"] = f"{project_root}/build-release:" + os.environ.get("LD_LIBRARY_PATH", "")


import numpy as np
import tvm
from tvm import auto_scheduler
from tvm.auto_scheduler import SketchPolicy

TARGET = tvm.target.Target("cuda")



[12:04:12] /root/work/tvm-ansor/src/target/target_kind.cc:181: Warning: Unable to detect CUDA version, default to "-arch=sm_50" instead


In [ ]:
import json
import os
from typing import Any, Dict, List, Optional

from tvm.auto_scheduler import SketchPolicy

from ..modules.task_paths import load_and_register_tasks
from ..modules.schedule_generator import ScheduleGenerator
from ..modules.symbolic_state_bridge import build_symbolic_state

OUTPUT_DIR = "examples"




def _to_builtin(x: Any) -> Any:
    if x is None or isinstance(x, (str, int, float, bool)):
        return x

    if isinstance(x, dict):
        return {str(k): _to_builtin(v) for k, v in x.items()}

    if isinstance(x, (list, tuple)):
        return [_to_builtin(v) for v in x]

    try:
        return [_to_builtin(v) for v in x]
    except TypeError:
        pass

    try:
        return int(x)
    except Exception:
        pass

    try:
        return float(x)
    except Exception:
        pass

    return str(x)


def _get_hw_info(task: Any) -> Dict[str, Any]:
    hp = getattr(task, "hardware_params", None)
    if hp is None:
        return {}

    keys = [
        "num_cores",
        "vector_unit_bytes",
        "cache_line_bytes",
        "max_shared_memory_per_block",
        "max_local_memory_per_block",
        "max_threads_per_block",
        "max_vthread_extent",
        "warp_size",
    ]
    out = {}
    for key in keys:
        if hasattr(hp, key):
            out[key] = _to_builtin(getattr(hp, key))
    return out


def _serialize_transform_step(step: Any, step_idx: int) -> Dict[str, Any]:
    attrs = [
        "stage_id",
        "iter_id",
        "target_stage_id",
        "target_iter_id",
        "src_step_id",
        "src_step_ids",
        "reader_stage_ids",
        "fused_ids",
        "after_ids",
        "lengths",
        "extent",
        "inner_to_outer",
        "annotation",
        "pragma_type",
        "offset",
        "level",
        "factor_or_nparts",
        "n_split",
    ]
    payload: Dict[str, Any] = {
        "step_idx": step_idx,
        "type": step.type_key.split(".")[-1],
    }
    for attr in attrs:
        if hasattr(step, attr):
            payload[attr] = _to_builtin(getattr(step, attr))
    return payload


def _choose_representative_split_param(gen: ScheduleGenerator) -> Optional[str]:
    full = gen.get_full_var_order_entries()
    for name in full["param_order"]:
        if name.startswith("sp_"):
            return name
    return None


def _choose_sample_candidate(gen: ScheduleGenerator, param_name: str) -> Optional[int]:
    report = gen.get_param_candidates(param_name)
    candidates = list(report.get("candidates", []))
    if not candidates:
        return None

    nontrivial = [x for x in candidates if int(x) > 1]
    if nontrivial:
        return int(nontrivial[min(len(nontrivial) // 2, len(nontrivial) - 1)])
    return int(candidates[0])


def _build_observability_snapshot(gen: ScheduleGenerator) -> Dict[str, Any]:
    full_var_order = gen.get_full_var_order_entries()
    representative = _choose_representative_split_param(gen)

    out: Dict[str, Any] = {
        "full_var_order": _to_builtin(full_var_order),
        "constraints_under_empty_assignment": _to_builtin(
            gen.get_constraints_under_assignment({}, include_vars=True, include_eval=True)
        ),
        "constraint_records": _to_builtin(gen._get_constraint_records()),
    }

    if representative is None:
        out["representative_param"] = None
        return out

    candidate_report = gen.get_param_candidates(representative)
    chosen = _choose_sample_candidate(gen, representative)

    out["representative_param"] = representative
    out["param_candidates"] = _to_builtin(candidate_report)
    out["propagation_example"] = None

    if chosen is not None:
        out["propagation_example"] = _to_builtin(
            gen.propagate_param_assignment(representative, chosen, {})
        )

    return out


def _build_task_meta(task: Any, task_index: int) -> Dict[str, Any]:
    target = getattr(task, "target", None)
    return {
        "task_index": task_index,
        "desc": str(getattr(task, "desc", "")),
        "workload_key": str(getattr(task, "workload_key", "")),
        "target_kind": None if target is None else str(getattr(target, "kind", "")),
        "target_model": None if target is None else str(getattr(target, "model", "")),
        "hardware_params": _get_hw_info(task),
        "num_ops": len(getattr(getattr(task, "compute_dag", None), "ops", [])),
    }


def _generate_concrete_sketches(task: Any) -> List[Any]:
    policy = SketchPolicy(
        task,
        params={"sample_init_no_invalid": 1},
        verbose=False,
    )
    return list(policy.generate_concrete_sketches())


def _build_example_payload(task: Any, task_index: int, state: Any, sketch_index: int) -> Dict[str, Any]:
    sym_state = build_symbolic_state(task, state)
    gen = ScheduleGenerator.from_task_state(task, state)

    payload = {
        "meta": {
            "task_index": task_index,
            "sketch_index": sketch_index,
            "note": "This JSON is TVM-free after export, but generating it requires TVM.",
        },
        "task": _build_task_meta(task, task_index),
        "transform_steps": [
            _serialize_transform_step(step, step_idx)
            for step_idx, step in enumerate(state.transform_steps)
        ],
        # 핵심 변경점: symbolic state를 string으로 저장
        "symbolic_state": sym_state.to_str(delete_trivial_loop=True),
        "sym_map": {str(k): _to_builtin(v) for k, v in sym_state.sym_map.items()},
        "observability": _build_observability_snapshot(gen),
    }
    return payload


def _build_output_path(task_index: int, sketch_index: int) -> str:
    return os.path.join(
        OUTPUT_DIR,
        f"t{task_index}_s{sketch_index}.json",
    )


In [10]:
# =========================
# hardcoded config
# =========================
NETWORK_INFO_FOLDER = "/root/work/tvm-ansor/gallery/dataset/network_info_all"
TASK_INDICES = [584, 1490]
MAX_SKETCHES_PER_TASK = 1

# =========================
# execute
# =========================
os.makedirs(OUTPUT_DIR, exist_ok=True)

tasks = load_and_register_tasks(NETWORK_INFO_FOLDER)

written_paths = []

for task_index in TASK_INDICES:
    if task_index < 0 or task_index >= len(tasks):
        raise ValueError(f"task index out of range: {task_index} (num_tasks={len(tasks)})")

    task = tasks[task_index]
    sketches = _generate_concrete_sketches(task)
    selected_sketches = sketches[:MAX_SKETCHES_PER_TASK]

    for sketch_index, state in enumerate(selected_sketches):
        payload = _build_example_payload(task, task_index, state, sketch_index)
        output_path = _build_output_path(task_index, sketch_index)

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)

        written_paths.append(output_path)
        print(f"wrote: {output_path}")

print("\nwritten files:")
for path in written_paths:
    print(path)

wrote: examples/t584_s0.json
wrote: examples/t1490_s0.json

written files:
examples/t584_s0.json
examples/t1490_s0.json
